In [ ]:
# ==========================================
# BLOCK 1: ARMORED INGESTION & FULL INTEGRITY AUDIT
# ==========================================
import pandas as pd
import os

print("Configuring paths and loading all sentiment datasets...")
data_path = "../data"

# 1. OS Pathing
orders = pd.read_csv(os.path.join(data_path, "olist_orders_dataset.csv"))
customers = pd.read_csv(os.path.join(data_path, "olist_customers_dataset.csv"))
order_items = pd.read_csv(os.path.join(data_path, "olist_order_items_dataset.csv"))
products = pd.read_csv(os.path.join(data_path, "olist_products_dataset.csv"))
reviews = pd.read_csv(os.path.join(data_path, "olist_order_reviews_dataset.csv"))

# 2. Strict ID Type Casting (The Anti-Crash Guard)
for df in [orders, customers, order_items, reviews]:
    if 'order_id' in df.columns: df['order_id'] = df['order_id'].astype(str)
    if 'customer_id' in df.columns: df['customer_id'] = df['customer_id'].astype(str)
customers['customer_unique_id'] = customers['customer_unique_id'].astype(str)
order_items['product_id'] = order_items['product_id'].astype(str)
products['product_id'] = products['product_id'].astype(str)
reviews['review_id'] = reviews['review_id'].astype(str)

# 3. Null Annihilation on Primary Keys ONLY 
print("\n--- NULL ANNIHILATION ---")
tables = {
    "Orders": (orders, ['order_id', 'customer_id']),
    "Customers": (customers, ['customer_id', 'customer_unique_id']),
    "Items": (order_items, ['order_id', 'product_id']),
    "Products": (products, ['product_id']),
    "Reviews": (reviews, ['review_id', 'order_id'])
}

for name, (df, keys) in tables.items():
    init_len = len(df)
    df.dropna(subset=keys, inplace=True)
    if init_len - len(df) > 0:
        print(f"-> Purged {init_len - len(df)} rows missing critical IDs in {name}")

# 4. Strict Deduplication (WITH COUNTS)
print("\n--- DEDUPLICATION ---")
init_ord, init_cust = len(orders), len(customers)
init_prod, init_items = len(products), len(order_items)
init_rev = len(reviews)

orders.drop_duplicates(subset=['order_id'], inplace=True)
customers.drop_duplicates(subset=['customer_id'], inplace=True)  # Preserves repeat buyers!
products.drop_duplicates(subset=['product_id'], inplace=True)
order_items.drop_duplicates(inplace=True) # Full row dedupe for line items
reviews.drop_duplicates(subset=['review_id', 'order_id'], inplace=True)

print(f"-> Dropped {init_ord - len(orders)} duplicate orders.")
print(f"-> Dropped {init_cust - len(customers)} duplicate customers.")
print(f"-> Dropped {init_prod - len(products)} duplicate products.")
print(f"-> Dropped {init_items - len(order_items)} duplicate items.")
print(f"-> Dropped {init_rev - len(reviews)} duplicate reviews.")

print("\nSentiment datasets armored, type-cast, and fully scrubbed!")

Configuring paths and loading all sentiment datasets...

--- NULL ANNIHILATION ---

--- DEDUPLICATION ---
-> Dropped 0 duplicate orders.
-> Dropped 0 duplicate customers.
-> Dropped 0 duplicate products.
-> Dropped 0 duplicate items.
-> Dropped 0 duplicate reviews.

Sentiment datasets armored, type-cast, and fully scrubbed!


In [ ]:
print("\n--- DATA SHAPES ---")
print(f"Orders: {orders.shape}")
print(f"Customers: {customers.shape}")
print(f"Order Items: {order_items.shape}")
print(f"Products: {products.shape}")
print(f"Reviews: {reviews.shape}")

print("\n--- CRITICAL COLUMNS ---")
print(f"Products cols: {products.columns.tolist()}")
print(f"Reviews cols: {reviews.columns.tolist()}")


--- DATA SHAPES ---
Orders: (99441, 8)
Customers: (99441, 5)
Order Items: (112650, 7)
Products: (32951, 9)
Reviews: (99224, 7)

--- CRITICAL COLUMNS ---
Products cols: ['product_id', 'product_category_name', 'product_name_lenght', 'product_description_lenght', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']
Reviews cols: ['review_id', 'order_id', 'review_score', 'review_comment_title', 'review_comment_message', 'review_creation_date', 'review_answer_timestamp']


In [ ]:
print("\n--- DUPLICATE CHECK ---")
review_dupes = reviews.duplicated(subset=['review_id', 'order_id']).sum()
print(f"Duplicate reviews found: {review_dupes}")
reviews = reviews.drop_duplicates(subset=['review_id', 'order_id'])


--- DUPLICATE CHECK ---
Duplicate reviews found: 0


In [ ]:
# Filter Orders for 'delivered' status only & Map customer_unique_id
delivered_orders = orders[orders["order_status"] == "delivered"].copy()

# Bring in customer_unique_id from the customers table
orders_merged = delivered_orders.merge(
    customers[["customer_id", "customer_unique_id"]],
    on="customer_id",
    how="inner"
)

print(f"Delivered orders shape: {orders_merged.shape}")
display(orders_merged.head())

Delivered orders shape: (96478, 9)


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,7c396fd4830fd04220f754e42b4e5bff
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00,af07308b275d755c9edb36a90c618231
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00,3a653a41f6f9fc3d2a113cf8398680e8
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00,7c142cf63193a1473d2e66489a9ae977
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00,72632f0f9dd73dfee390c9b22eb56dd6


In [ ]:
# Build Top Category Feature (Item-Level Purchase Counts)
customer_items = orders_merged[["order_id", "customer_unique_id"]].merge(
    order_items[["order_id", "product_id"]],
    on="order_id",
    how="inner"
)

display(customer_items.head())

customer_products = customer_items.merge(
    products[["product_id", "product_category_name"]],
    on="product_id",
    how="left"
)

display(customer_products.head())


,order_id,customer_unique_id,product_id
0,e481f51cbdc54678b7cc49136f2d6af7,7c396fd4830fd04220f754e42b4e5bff,87285b34884572647811a353c7ac498a
1,53cdb2fc8bc7dce0b6741e2150273451,af07308b275d755c9edb36a90c618231,595fac2a385ac33a80bd5114aec74eb8
2,47770eb9100c2d0c44946d9cf07ec65d,3a653a41f6f9fc3d2a113cf8398680e8,aa4383b373c6aca5d8797843e5594415
3,949d5b44dbf5de918fe9c16f97b45f8a,7c142cf63193a1473d2e66489a9ae977,d0b61bfb1de832b15ba9d266ca96e5b0
4,ad21c59c0840e6cb83a9ceb5573f8159,72632f0f9dd73dfee390c9b22eb56dd6,65266b2da20d04dbe00c5c2d3bb7859e


,order_id,customer_unique_id,product_id,product_category_name
0,e481f51cbdc54678b7cc49136f2d6af7,7c396fd4830fd04220f754e42b4e5bff,87285b34884572647811a353c7ac498a,utilidades_domesticas
1,53cdb2fc8bc7dce0b6741e2150273451,af07308b275d755c9edb36a90c618231,595fac2a385ac33a80bd5114aec74eb8,perfumaria
2,47770eb9100c2d0c44946d9cf07ec65d,3a653a41f6f9fc3d2a113cf8398680e8,aa4383b373c6aca5d8797843e5594415,automotivo
3,949d5b44dbf5de918fe9c16f97b45f8a,7c142cf63193a1473d2e66489a9ae977,d0b61bfb1de832b15ba9d266ca96e5b0,pet_shop
4,ad21c59c0840e6cb83a9ceb5573f8159,72632f0f9dd73dfee390c9b22eb56dd6,65266b2da20d04dbe00c5c2d3bb7859e,papelaria


In [ ]:
# Handle missing category names gracefully instead of silent dropouts
customer_products["product_category_name"] = customer_products["product_category_name"].fillna("unknown_category")

category_counts = (
    customer_products
    .groupby(["customer_unique_id", "product_category_name"])
    .size()
    .reset_index(name="purchase_count")
)

# Sort and get the top category per unique customer
category_counts = category_counts.sort_values(
    ["customer_unique_id", "purchase_count"],
    ascending=[True, False]
)

display(category_counts.head())

top_category = category_counts.drop_duplicates(subset="customer_unique_id")
top_category = top_category[["customer_unique_id", "product_category_name"]].rename(
    columns={"product_category_name": "Top_Category"}
)

top_category.head()

,customer_unique_id,product_category_name,purchase_count
0,0000366f3b9a7992bf8c76cfdf3221e2,cama_mesa_banho,1
1,0000b849f77a49e4a4ce2b2a4ca5be3f,beleza_saude,1
2,0000f46a3911fa3c0805444483337064,papelaria,1
3,0000f6ccb0745a6a4b88665a16c9f078,telefonia,1
4,0004aac84e0df4da2b147fca70cf8255,telefonia,1


,customer_unique_id,Top_Category
0,0000366f3b9a7992bf8c76cfdf3221e2,cama_mesa_banho
1,0000b849f77a49e4a4ce2b2a4ca5be3f,beleza_saude
2,0000f46a3911fa3c0805444483337064,papelaria
3,0000f6ccb0745a6a4b88665a16c9f078,telefonia
4,0004aac84e0df4da2b147fca70cf8255,telefonia


In [ ]:
#Build Review / Sentiment Risk Feature (CORRECTED)
customer_reviews = orders_merged[["order_id", "customer_unique_id"]].merge(
    reviews[["order_id", "review_score"]],
    on="order_id",
    how="left"
)

review_summary = (
    customer_reviews
    .groupby("customer_unique_id")["review_score"]
    .agg(
        avg_review_score="mean",
        min_review_score="min",  # Grab their absolute worst experience
        review_count="count"
    )
    .reset_index()
)

# Safely handle nulls BEFORE the math
# If they never left a review, assume they are an "average" 3-star user
review_summary["min_review_score"] = review_summary["min_review_score"].fillna(3)
review_summary["avg_review_score"] = review_summary["avg_review_score"].fillna(3)

# Flag as churn risk if they have EVER left a 1 or 2 star review
review_summary["low_review_flag"] = (
    review_summary["min_review_score"] <= 2
).astype(int)

# Drop the min column to keep the final output clean
review_summary = review_summary.drop(columns=["min_review_score"])

print(f"Review summary shape: {review_summary.shape}")
display(review_summary.head())

Review summary shape: (93358, 4)


,customer_unique_id,avg_review_score,review_count,low_review_flag
0,0000366f3b9a7992bf8c76cfdf3221e2,5.0,1,0
1,0000b849f77a49e4a4ce2b2a4ca5be3f,4.0,1,0
2,0000f46a3911fa3c0805444483337064,3.0,1,0
3,0000f6ccb0745a6a4b88665a16c9f078,4.0,1,0
4,0004aac84e0df4da2b147fca70cf8255,5.0,1,0


In [ ]:

# Final Merge (Using Outer Join to prevent customer loss)
member3_features = top_category.merge(
    review_summary,
    on="customer_unique_id",
    how="outer"
)

# Fill nulls post-merge for safe downstream use (e.g., K-Means modeling)
member3_features["Top_Category"] = member3_features["Top_Category"].fillna("unknown_category")
member3_features["avg_review_score"] = member3_features["avg_review_score"].fillna(member3_features["avg_review_score"].mean())
member3_features["review_count"] = member3_features["review_count"].fillna(0)
member3_features["low_review_flag"] = member3_features["low_review_flag"].fillna(0).astype(int)

member3_features.head(20)

,customer_unique_id,Top_Category,avg_review_score,review_count,low_review_flag
0,0000366f3b9a7992bf8c76cfdf3221e2,cama_mesa_banho,5.0,1,0
1,0000b849f77a49e4a4ce2b2a4ca5be3f,beleza_saude,4.0,1,0
2,0000f46a3911fa3c0805444483337064,papelaria,3.0,1,0
3,0000f6ccb0745a6a4b88665a16c9f078,telefonia,4.0,1,0
4,0004aac84e0df4da2b147fca70cf8255,telefonia,5.0,1,0
5,0004bd2a26a76fe21f786e4fbd80607f,ferramentas_jardim,4.0,1,0
6,00050ab1314c0e55a6ca13cf7181fecf,telefonia,4.0,1,0
7,00053a61a98854899e70ed204dd4bafe,esporte_lazer,1.0,1,1
8,0005e1862207bf6ccc02e4228effd9a0,fashion_bolsas_e_acessorios,4.0,1,0
9,0005ef4cd20d2893f0d9fbd94d3c0d97,esporte_lazer,1.0,1,1


In [ ]:
# ==========================================
# POST-MERGE AUDIT & ARMORED EXPORT
# ==========================================
print("--- FINAL INTEGRITY AUDIT ---")

# Verify absolute uniqueness of the final output table
duplicate_users = member3_features["customer_unique_id"].duplicated().sum()
if duplicate_users > 0:
    raise ValueError(f"CRITICAL: Found {duplicate_users} duplicate humans in the final output. Aggregation failed.")
else:
    print("-> User uniqueness verified. 1 Human = 1 Row.")

print("\nExporting Sentiment & Category Metrics...")

# The Directory Defense
output_dir = "../data"
os.makedirs(output_dir, exist_ok=True)
output_file = os.path.join(output_dir, "member3_features.csv")

# Save directly using the os path
member3_features.to_csv(output_file, index=False)

print(f"Success! Member 3 features saved to: {output_file}")

--- FINAL INTEGRITY AUDIT ---
-> User uniqueness verified. 1 Human = 1 Row.

Exporting Sentiment & Category Metrics...
Success! Member 3 features saved to: ../data\member3_features.csv
